[Project Stone]
- 돌 분류 프로젝트

In [2]:
import sys
import torch
import os

print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\ProgramData\anaconda3\envs\DL_P310\python.exe
Python Version: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.4.0
PyTorch CUDA Build: 12.4
CUDA Available: True
CUDA Version (Runtime): 12.4
Device Name: NVIDIA A100 80GB PCIe
Device Compute Capability: (8, 0)
-------------------------


In [3]:
import pandas as pd
import numpy as np

In [4]:
numlist = os.listdir('./_data/open/train')
trainDIR = './_data/open/train'
sum = 0

for i in numlist:
    a = (len(os.listdir('./_data/open/train/'+i)))
    print(i, a)
    sum += a
print(sum)

Andesite 43802
Basalt 26810
Gneiss 73914
Granite 92923
Mud_Sandstone 89467
Weathered_Rock 37169
364085


In [5]:
numlist = os.listdir('./_data/open/test')
testDIR = './_data/open/test'
len(numlist)

95006

In [6]:
## 작업순서
## 데이테 셋 만들기 
## 데이터 로더 만들기
## 폴더별로 되있으니까 imgeafolder사용하면 될듯?


In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split # train, valid 나누기용


from PIL import Image
from sklearn.metrics import f1_score
from torchvision import transforms

In [8]:
IMG_SIZE = 224  # ViT-B/16 기준

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [9]:
classes = ['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock']
classes2idx = {x:idx for idx,x in enumerate(classes)}
classes2idx

{'Andesite': 0,
 'Basalt': 1,
 'Gneiss': 2,
 'Granite': 3,
 'Mud_Sandstone': 4,
 'Weathered_Rock': 5}

In [10]:
class CustomImageFolder(ImageFolder):
    def __init__(self, root, classes2idx):
        self.classes2idx = classes2idx
        self.class_to_idx = classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root)

    def find_classes(self, directory):
        # 고정된 클래스 사용
        return self.classes, self.classes2idx

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)
        if self.target_transform is not None:
            target = self.target_transform(target)

        # 필요시 클래스명까지 리턴
        class_name = self.classes[target]
        return sample, target  # 또는 (sample, target, class_name) 도 가능

In [11]:
originDS = CustomImageFolder(trainDIR, classes2idx=classes2idx)

In [12]:
originDS.class_to_idx

{'Andesite': 0,
 'Basalt': 1,
 'Gneiss': 2,
 'Granite': 3,
 'Mud_Sandstone': 4,
 'Weathered_Rock': 5}

In [13]:
class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None, classes2idx=None):
        """
        Args:
            subset (torch.utils.data.Subset): random_split으로 분할된 Subset 객체
            transform (callable, optional): 샘플에 적용될 선택적 변환
        """
        self.subset = subset
        self.transform = transform
        self.classes2idx = classes2idx

    def __getitem__(self, index):
        # Subset으로부터 원본 데이터 (이미지, 레이블)를 가져옴
        x, y = self.subset[index]
        if self.transform:
            # 이미지에만 변환 적용
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [14]:

# --- 데이터 분할 및 각기 다른 변환 적용 ---

# 1. 원본 데이터셋을 train과 validation Subset으로 분할
total_size = len(originDS)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# generator를 사용하여 재현 가능한 분할을 할 수 있습니다.
generator = torch.Generator().manual_seed(42) # 시드 고정
train_subset, valid_subset = random_split(originDS, [train_size, valid_size], generator=generator)

# 2. 분할된 Subset에 각각 다른 transform을 적용하는 TransformedDataset 생성
trainDS = TransformedDataset(train_subset, transform=train_transform, classes2idx=classes2idx)
validDS = TransformedDataset(valid_subset, transform=val_transform, classes2idx=classes2idx)

In [15]:
print('train', len(trainDS))
# print(trainDS.classes)
trainDS.classes2idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 291268


{0: 'Andesite',
 1: 'Basalt',
 2: 'Gneiss',
 3: 'Granite',
 4: 'Mud_Sandstone',
 5: 'Weathered_Rock'}

In [16]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
DEVICE

'cuda'

In [17]:
trainDL = DataLoader(
    trainDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    validDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [18]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn

num_classes = len(classes2idx)
num_classes

6

In [19]:
# ✅ 모델: ResNet101 + 마지막 fc 교체
model = models.resnet101(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load('./_model/RES/RES_3_0305_f1_0.7637.pth'))
model = model.to(DEVICE)


c:\ProgramData\anaconda3\envs\DL_P310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\DL_P310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
C:\Users\K\AppData\Local\Temp\ipykernel_9232\791383286.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_

In [20]:
# from typing import Union

# class FocalLoss(nn.Module):
#     """
#     Focal Loss for multi-class classification.
#     논문: Lin et al., Focal Loss for Dense Object Detection (https://arxiv.org/abs/1708.02002)

#     Args:
#         alpha (float or Tensor or None): 클래스별 가중치.
#             - float: 모든 클래스에 동일하게 적용 (ex. 0.25)
#             - Tensor: 클래스 수와 동일한 길이의 가중치 벡터
#             - None: 균등 가중치
#         gamma (float): Focusing parameter (default=2.0).
#         reduction (str): 'mean', 'sum', or 'none' (default='mean').
#     """
#     def __init__(self, alpha: Union[float, torch.Tensor, None] = 0.25,
#                  gamma: float = 2.0, reduction: str = 'mean'):
#         super(FocalLoss, self).__init__()
#         self.gamma = gamma
#         self.reduction = reduction

#         # alpha 등록
#         if alpha is not None:
#             if isinstance(alpha, float):
#                 self.register_buffer('alpha', torch.tensor([alpha], dtype=torch.float))
#             elif isinstance(alpha, torch.Tensor):
#                 if alpha.ndim != 1:
#                     raise ValueError("alpha tensor must be 1D")
#                 self.register_buffer('alpha', alpha.clone().detach())
#             else:
#                 raise TypeError(f"Unsupported alpha type: {type(alpha)}")
#         else:
#             self.register_buffer('alpha', None)

#     def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
#         if logits.ndim != 2:
#             raise ValueError(f"Expected logits shape [B, C], got {logits.shape}")
#         if targets.ndim != 1:
#             raise ValueError(f"Expected targets shape [B], got {targets.shape}")

#         device = logits.device
#         num_classes = logits.size(1)
#         targets = targets.to(device).long()

#         # log_softmax로 수치 안정성 확보
#         log_probs = F.log_softmax(logits, dim=1)
#         log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
#         pt = log_pt.exp()

#         # alpha 가중치 적용
#         if self.alpha is not None:
#             if self.alpha.numel() == 1:
#                 alpha_t = self.alpha.to(device).expand_as(pt)
#             else:
#                 if self.alpha.numel() != num_classes:
#                     raise ValueError(f"alpha length {self.alpha.numel()} ≠ num_classes {num_classes}")
#                 alpha_t = self.alpha.to(device).gather(0, targets)
#         else:
#             alpha_t = torch.ones_like(pt)

#         loss = -alpha_t * (1 - pt) ** self.gamma * log_pt

#         # Reduction 적용
#         if self.reduction == 'mean':
#             return loss.mean()
#         elif self.reduction == 'sum':
#             return loss.sum()
#         elif self.reduction == 'none':
#             return loss
#         else:
#             raise ValueError(f"Invalid reduction type: {self.reduction}")


In [21]:
from torch import optim
from tqdm import tqdm
# ✅ 손실함수, 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [22]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [23]:
import time

In [24]:
now = time.localtime()
ct = time.strftime("%y.%m.%d %H:%M:%S",now)
ct

'25.05.14 07:51:06'

In [ ]:

# 하이퍼파라미터 설정 (사용자 코드에서 가져옴)
EPOCH = 10 # 예시 값, 실제 값으로 대체 필요
best_val_loss = float('inf')
best_val_f1 = 0.0
patience = 3
patience_counter = 0

# 에포크별 '평균' 지표를 저장할 리스트
train_metrics = []
val_metrics = []

# psutil.cpu_percent() 초기 호출 (첫 호출 시 의미 없는 값 반환 방지)
psutil.cpu_percent(interval=None)
time.sleep(0.1) # CPU 사용률 측정을 위한 짧은 대기

# scaler 초기화 (사용자 코드에 이미 존재) - CUDA 사용 가능할 때만 활성화하도록 수정
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


for epoch in range(EPOCH):
    model.train()
    epoch_train_loss_sum = 0.0
    epoch_train_correct = 0
    epoch_train_total = 0
    epoch_train_cpu_sum = 0.0
    epoch_train_gpu_sum = 0.0
    train_batches = 0

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # AMP 적용: autocast 컨텍스트 내에서 순전파 및 손실 계산
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        # AMP 적용: GradScaler를 사용하여 손실 스케일링 및 역전파
        scaler.scale(loss).backward()
        # AMP 적용: GradScaler를 사용하여 옵티마이저 스텝 및 그래디언트 언스케일링
        scaler.step(optimizer)
        # AMP 적용: 다음 반복을 위해 스케일러 업데이트
        scaler.update()

        epoch_train_loss_sum += loss.item()
        _, predicted = outputs.max(1)
        epoch_train_correct += predicted.eq(labels).sum().item()
        epoch_train_total += labels.size(0)
        train_batches += 1

        cpu_usage = psutil.cpu_percent(interval=None)
        gpu_util, _, _ = get_gpu_usage() # 이 함수는 사용자 코드에 정의되어 있어야 합니다.

        epoch_train_cpu_sum += cpu_usage
        epoch_train_gpu_sum += gpu_util if gpu_util is not None else 0

        current_cumulative_acc = 100. * epoch_train_correct / epoch_train_total
        train_loop.set_postfix(loss=loss.item(),
                                 acc=f"{current_cumulative_acc:.2f}%",
                                 cpu=f"{cpu_usage:.1f}%",
                                 gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    if train_batches > 0:
        avg_train_loss = epoch_train_loss_sum / train_batches
        avg_train_acc = 100. * epoch_train_correct / epoch_train_total
        avg_train_cpu = epoch_train_cpu_sum / train_batches
        avg_train_gpu = epoch_train_gpu_sum / train_batches
    else:
        avg_train_loss, avg_train_acc, avg_train_cpu, avg_train_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    train_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        "accuracy": avg_train_acc,
        "avg_cpu_percent": avg_train_cpu,
        "avg_gpu_percent": avg_train_gpu
    })

    # ----------- Validation -----------
    model.eval()
    epoch_val_loss_sum = 0.0
    epoch_val_correct = 0
    epoch_val_total = 0
    epoch_val_cpu_sum = 0.0
    epoch_val_gpu_sum = 0.0
    val_batches = 0

    epoch_val_preds = []
    epoch_val_labels = []

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Validation", leave=False)

    with torch.no_grad(): # 그래디언트 계산 비활성화
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            # AMP 적용 (선택 사항이지만, 학습과 일관성을 위해 또는 검증 속도 향상을 위해 사용 가능)
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            epoch_val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            epoch_val_correct += predicted.eq(labels).sum().item()
            epoch_val_total += labels.size(0)
            val_batches += 1

            epoch_val_preds.extend(predicted.cpu().numpy())
            epoch_val_labels.extend(labels.cpu().numpy())

            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, _, _ = get_gpu_usage() # 이 함수는 사용자 코드에 정의되어 있어야 합니다.

            epoch_val_cpu_sum += cpu_usage
            epoch_val_gpu_sum += gpu_util if gpu_util is not None else 0

            current_cumulative_acc = 100. * epoch_val_correct / epoch_val_total
            val_loop.set_postfix(loss=loss.item(),
                                   acc=f"{current_cumulative_acc:.2f}%",
                                   cpu=f"{cpu_usage:.1f}%",
                                   gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    if val_batches > 0:
        avg_val_loss = epoch_val_loss_sum / val_batches
        avg_val_acc = 100. * epoch_val_correct / epoch_val_total
        avg_val_cpu = epoch_val_cpu_sum / val_batches
        avg_val_gpu = epoch_val_gpu_sum / val_batches
        # F1 스코어 계산 시 epoch_val_labels 또는 epoch_val_preds가 비어있으면 오류 발생 가능성 있음
        if epoch_val_labels and epoch_val_preds:
             avg_val_f1 = f1_score(epoch_val_labels, epoch_val_preds, average='macro', zero_division=0)
        else:
             avg_val_f1 = 0.0
    else:
        avg_val_loss, avg_val_acc, avg_val_cpu, avg_val_gpu, avg_val_f1 = 0, 0, 0, 0, 0

    val_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_val_loss,
        "accuracy": avg_val_acc,
        "f1_score": avg_val_f1,
        "avg_cpu_percent": avg_val_cpu,
        "avg_gpu_percent": avg_val_gpu
    })

    # --- 결과 출력 및 모델 저장 로직 ---
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | Avg CPU: {avg_train_cpu:.1f}%, Avg GPU: {avg_train_gpu:.1f}%")
    print(f"  Valid Loss: {avg_val_loss:.4f}, Valid Acc: {avg_val_acc:.2f}% | Avg CPU: {avg_val_cpu:.1f}%, Avg GPU: {avg_val_gpu:.1f}%")
    print(f"  Valid F1: {avg_val_f1:.4f}")

    now = time.localtime()
    ct = time.strftime("%H%M", now)

    # Early Stopping 및 모델 저장
    if avg_val_f1 > best_val_f1:
        print(f"  Validation F1 improved ({best_val_f1:.4f} --> {avg_val_f1:.4f}). Saving model...")
        best_val_f1 = avg_val_f1
        save_dir = "./_model/RES/"
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"RES_{epoch+1}_{ct}_f1_{avg_val_f1:.4f}.pth")
        try:
            torch.save(model.state_dict(), save_path)
            print(f"  Model saved to {save_path}")
        except Exception as e:
            print(f"  Warning: Failed to save model. Error: {e}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  Validation F1 did not improve from {best_val_f1:.4f}. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch + 1} epochs.")
            break

    # 지표를 CSV로 저장 (루프 안에서 각 에포크의 누적 결과를 저장)
    # 파일명에 ct (시간)이 포함되어, 실행 시점 기준으로 파일이 생성되고, 해당 파일에 에포크별 결과가 누적됩니다.
    train_df = pd.DataFrame(train_metrics)
    val_df = pd.DataFrame(val_metrics)
    
    # CSV 저장 경로에 디렉토리가 없으면 생성
    metrics_save_dir = "./_data/metrics_RES/"
    os.makedirs(metrics_save_dir, exist_ok=True)
    
    train_df.to_csv(os.path.join(metrics_save_dir, f"train_metrics_VIT_{ct}.csv"), index=False)
    val_df.to_csv(os.path.join(metrics_save_dir, f"val_metrics_VIT_{ct}.csv"), index=False)

# 학습 완료 후 저장된 지표 확인
print("\n--- Training Metrics (Epoch Averages) ---")
for metrics in train_metrics: # i 변수 제거
    print(f"Epoch {metrics['epoch']}: Loss={metrics['loss']:.4f}, Acc={metrics['accuracy']:.2f}%, CPU={metrics['avg_cpu_percent']:.1f}%, GPU={metrics['avg_gpu_percent']:.1f}%")

print("\n--- Validation Metrics (Epoch Averages) ---")
for metrics in val_metrics: # i 변수 제거
    print(f"Epoch {metrics['epoch']}: Loss={metrics['loss']:.4f}, Acc={metrics['accuracy']:.2f}%, F1={metrics['f1_score']:.4f}, CPU={metrics['avg_cpu_percent']:.1f}%, GPU={metrics['avg_gpu_percent']:.1f}%")


C:\Users\K\AppData\Local\Temp\ipykernel_14888\2728392874.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
[Epoch 1/10] Training:   0%|          | 0/9102 [00:00<?, ?it/s]C:\Users\K\AppData\Local\Temp\ipykernel_14888\2728392874.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


KeyboardInterrupt: 

In [27]:
sampleDF = pd.DataFrame(pd.read_csv('./_data/open/sample_submission.csv'))
print(sampleDF.head())
testDF = pd.DataFrame(pd.read_csv('./_data/open/test.csv'))
print(testDF.head())

           ID rock_type
0  TEST_00000       Etc
1  TEST_00001       Etc
2  TEST_00002       Etc
3  TEST_00003       Etc
4  TEST_00004       Etc
           ID               img_path
0  TEST_00000  ./test/TEST_00000.jpg
1  TEST_00001  ./test/TEST_00001.jpg
2  TEST_00002  ./test/TEST_00002.jpg
3  TEST_00003  ./test/TEST_00003.jpg
4  TEST_00004  ./test/TEST_00004.jpg


In [28]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [29]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [30]:
testDS = TestImageDataset(testDF, image_root="./_data/open/", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

In [32]:
# model.eval()
# predictions = []

# with torch.no_grad():
#     count = 0
#     for images, filenames in testDL:
#         images = images.to(DEVICE)
#         outputs = model(images)
#         preds = outputs.argmax(dim=1).cpu().numpy()
        
#         for fname, pred in zip(filenames, preds):
#             predictions.append((fname, idx2class[pred.item()]))
#         count += 1
#         # if count ==3: break

# Softmax 확률 임계값 설정
CONFIDENCE_THRESHOLD = 0.5
# 임계값 미달 시 사용할 라벨 문자열
ETC_LABEL_STRING = "Etc"

print(f"Softmax 확률 임계값: {CONFIDENCE_THRESHOLD}")
print(f"임계값 미달 시 분류될 라벨: '{ETC_LABEL_STRING}'")

model.eval()  # 모델을 평가 모드로 설정
predictions = [] # (파일명, 예측된_클래스_이름_문자열) 튜플을 저장할 리스트

with torch.no_grad():  # 그래디언트 계산 비활성화
    # count = 0 # 테스트용 카운터 (필요시 주석 해제)
    for images, filenames in testDL:
        images = images.to(DEVICE)  # 이미지를 설정된 장치(GPU 또는 CPU)로 이동

        outputs = model(images)  # 모델의 순전파 실행 (일반적으로 logits 반환)

        # Softmax 함수를 적용하여 각 클래스에 대한 확률 계산
        probabilities = F.softmax(outputs, dim=1)

        # 각 샘플에 대해 가장 높은 확률(신뢰도)과 해당 클래스의 인덱스 가져오기
        max_probabilities, predicted_indices = torch.max(probabilities, dim=1)

        # 결과를 CPU로 옮기고 리스트 형태로 변환 (또는 NumPy 배열)
        max_probs_list = max_probabilities.cpu().tolist()
        predicted_indices_list = predicted_indices.cpu().tolist()

        for i in range(len(filenames)):
            filename = filenames[i]
            confidence = max_probs_list[i]
            predicted_idx = predicted_indices_list[i]

            final_predicted_label = ""
            if confidence < CONFIDENCE_THRESHOLD:
                # 가장 높은 확률(신뢰도)이 임계값 미만이면 "Etc"로 분류
                final_predicted_label = ETC_LABEL_STRING
            else:
                # 신뢰도가 임계값 이상이면, 원래 예측된 클래스 이름 사용
                # idx2class는 학습된 클래스 인덱스만 가지고 있어야 합니다.
                if predicted_idx in idx2class:
                    final_predicted_label = idx2class[predicted_idx]
                else:
                    # 이 경우는 모델 출력 인덱스가 idx2class에 없는 예외적인 상황
                    print(f"경고: 파일 '{filename}'의 예측 인덱스 '{predicted_idx}'가 idx2class에 없습니다. '{ETC_LABEL_STRING}'로 처리합니다.")
                    final_predicted_label = ETC_LABEL_STRING
            
            predictions.append((filename, final_predicted_label))

        # count += 1 # 테스트용 카운터
        # if count == 3: break # 테스트용: 처음 몇 배치만 처리

Softmax 확률 임계값: 0.5
임계값 미달 시 분류될 라벨: 'Etc'


In [37]:
sampleDF['rock_type'].value_counts()

rock_type
Mud_Sandstone     52789
Granite           17707
Basalt             8554
Etc                8513
Weathered_Rock     2875
Andesite           2378
Gneiss             2190
Name: count, dtype: int64

In [ ]:
# for i in range(sampleDF.shape[0]):
for i in range(95006):
    if sampleDF.loc[i,'ID'] == predictions[i][0]:
        sampleDF.loc[i,'rock_type'] = predictions[i][1]
    else:
        continue
sampleDF['rock_type'].value_counts()

,ID,rock_type
0,TEST_00000,Mud_Sandstone
1,TEST_00001,Mud_Sandstone
2,TEST_00002,Mud_Sandstone
3,TEST_00003,Granite
4,TEST_00004,Granite
5,TEST_00005,Etc
6,TEST_00006,Etc
7,TEST_00007,Granite
8,TEST_00008,Gneiss
9,TEST_00009,Granite


In [36]:
now = time.localtime()
ct = time.strftime("%H%M",now)
csv_path = './_data/open/sample_submission_answer_RES_%s.csv'%ct
try:
    sampleDF.to_csv(csv_path, index=False)
    print(csv_path, 'saved')
except:
    print('failed')

./_data/open/sample_submission_answer_RES_0809.csv saved
